# BigAlpha Stage A dual-frequency frozen adapter

This notebook loads `weights.json`, uses the official 1-minute and 5-minute raw tables, and returns `date`, `instrument`, `score`.

The Tier-1 one-minute champion is frozen inside each saved Stage-A model. The trainable addition is an A5-Lite raw-field, three-level DeepLOB-style causal encoder and one last-token cross-attention residual adapter over the final 12 encoded 5m tokens. No derived input feature is constructed.

In [ ]:
import os

import dai
import numpy as np
import pandas as pd
import structlog
import torch

from train import (
    MODEL_PATH,
    load_stage_a_ensemble,
    pool,
    predict_stage_a_scores,
    train_and_save,
)

logger = structlog.get_logger()


def main(datasources, start_date, end_date):
    """Return exact competition columns: date, instrument, score."""
    table1m = datasources.get("bar1m", "bigalpha_2026_stock_bar1m")
    table5m = datasources.get("bar5m", "bigalpha_2026_stock_bar5m")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"Missing {MODEL_PATH}; run Stage A training or upload weights.json."
        )

    checkpoint, models = load_stage_a_ensemble(
        MODEL_PATH, map_location=device
    )
    stats1 = (
        np.asarray(checkpoint["mean_1m"], np.float32),
        np.asarray(checkpoint["std_1m"], np.float32),
    )
    stats5 = (
        np.asarray(checkpoint["mean_5m"], np.float32),
        np.asarray(checkpoint["std_5m"], np.float32),
    )
    predictions = predict_stage_a_scores(
        models,
        table1m,
        table5m,
        start_date,
        end_date,
        pool(start_date, end_date),
        stats1,
        stats5,
        device,
    )

    official = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    result = (
        pd.merge(predictions, official, on=["date", "instrument"], how="inner")
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["score"])
        .drop_duplicates(["date", "instrument"])
        [["date", "instrument", "score"]]
        .reset_index(drop=True)
    )
    logger.info(
        "stage A scores complete",
        rows=len(result),
        days=result["date"].nunique(),
        instruments=result["instrument"].nunique(),
    )
    return result


# Development only. Keep False in the submission notebook.
RUN_EXAMPLE = False
if RUN_EXAMPLE:
    from bigmodule import M

    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
        "bar5m": "bigalpha_2026_stock_bar5m",
    }
    if not os.path.exists(MODEL_PATH):
        train_and_save(datasources)
    score_data = main(
        datasources,
        "2024-01-02 00:00:00",
        "2024-01-12 23:59:59",
    )
    result = M.bigalpha_eval._latest(
        factor_data=score_data,
        show=True,
    )
